# 04 GAT readiness model on Junyi (C1 Adaptive Curriculum Engine)

Trains the Graph Attention Network that scores how ready a learner is for a concept, from their BKT mastery of the concept and of its prerequisites. The trained model is exported as `gat-v2.npz` and runs with numpy inside `curriculum-service`.

**Task.** For every answer *t*: predict whether it is correct, from features computed with answers before *t* only.

**Graph-agnostic features.** Each node (exercise) gets 6 numbers that exist in any prerequisite graph, so a model trained on Junyi can score the Java PF + DSA graph:

| Feature | Meaning |
|---|---|
| `mastery` | BKT probability the node is learned (notebook 03 parameters) |
| `log_evidence` | log(1 + answers on this node so far) |
| `log_depth` | log(1 + longest prerequisite chain below the node) |
| `log_prereq_count` | log(1 + number of prerequisites) |
| `mean_prereq_mastery` | mean mastery of its prerequisites (1.0 when it has none) |
| `bkt_p_correct` | BKT's own predicted chance of a correct answer: mastery x (1 - slip) + (1 - mastery) x guess, with that node's BKT values |

**Learner features (v2).** Three numbers about the learner, not about a node, so they also exist on any platform: `overall_correct_rate` and `recent_correct_rate` (last 5 answers, any exercise), both smoothed as (correct + 1) / (answered + 2), and `previous_correct` (0.5 before the first answer). They give the model the general ability and momentum signal that DKT and GKT get from the whole history (v1 without them: test AUC 0.7835 vs DKT 0.7877 and GKT 0.7885 in notebook 05). They enter the classifier, not the attention, so the explanation still comes from the prerequisite attention alone.

**Model.** One GAT layer over the target node and its direct prerequisites (with a self-loop), multi-head attention, then a small classifier that outputs a **correction to BKT**: logit(P(correct)) = logit(`bkt_p_correct` of the target) + correction. The correction layer starts at zero, so before training the model equals BKT exactly, and it only moves away where the prerequisite graph helps. (A first version without `bkt_p_correct` scored below BKT, AUC 0.756 vs 0.785, because it could not see each exercise's guess and slip.) The attention weights over the prerequisite edges feed the explanation: weak prerequisite = argmax of attention x (1 - mastery).

**Evaluation.** 5-fold cross-validation over the training students (GAT vs BKT-only on the same folds, paired t-test), then one final model on all training students, scored on the held-out test students.

Needs notebooks 01, 02 and 03. Use a **GPU runtime** (T4): Runtime > Change runtime type.

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training and 1,000 test students, 1 epoch, for a fast check.
QUICK = False

In [2]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "gat"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (PROCESSED / "interactions.parquet").exists(), "Run notebook 01 first (or upload its outputs)."
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Load answers, splits, the Junyi graph and the BKT parameters

In [3]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CONFIG = {"SEED": SEED, "QUICK": QUICK, "HEADS": 4, "HIDDEN": 16, "CLASSIFIER": 32,
          "EPOCHS": 1 if QUICK else 5, "BATCH": 4096, "LR": 3e-3, "WEIGHT_DECAY": 1e-5,
          "LEAKY_SLOPE": 0.2, "MODEL_VERSION": "gat-v2"}

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise_idx", "step", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")  # row 0 prerequisite, row 1 exercise
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise")

fold_of = {u: int(k) for k, users in splits["folds"].items() for u in users}
test_ids = set(splits["test"])
if QUICK:
    keep = set(rng.choice(sorted(fold_of), 3_000, replace=False)) | set(rng.choice(sorted(test_ids), 1_000, replace=False))
    inter = inter[inter.user_id.isin(keep)].reset_index(drop=True)
print(f"{inter.user_id.nunique():,} students, {len(inter):,} answers")

device: cuda
30,000 students, 5,959,373 answers


In [4]:
n_ex = len(exercise_index)
names = sorted(exercise_index, key=exercise_index.get)
prior = bkt.loc[names, "p_init"].to_numpy()
learn = bkt.loc[names, "p_learn"].to_numpy()
guess = bkt.loc[names, "p_guess"].to_numpy()
slip = bkt.loc[names, "p_slip"].to_numpy()

G = nx.DiGraph()
G.add_nodes_from(range(n_ex))
G.add_edges_from(zip(edge_index[0].tolist(), edge_index[1].tolist()))
assert nx.is_directed_acyclic_graph(G)
depth = np.zeros(n_ex)
for v in nx.topological_sort(G):
    preds = list(G.predecessors(v))
    depth[v] = 1 + max(depth[p] for p in preds) if preds else 0
prereqs = [sorted(G.predecessors(v)) for v in range(n_ex)]
K = max(len(p) for p in prereqs)
print(f"max prerequisites per exercise K = {K}, longest chain = {int(depth.max())}")
pd.Series([len(p) for p in prereqs]).value_counts().sort_index()

max prerequisites per exercise K = 7, longest chain = 65


,count
0,69
1,535
2,138
3,24
4,4
5,4
6,1
7,1


## 2. Features from answers before *t* only

One pass over every student's answers in time order. At answer *t* the features are read first, then the answer updates BKT, so nothing from *t* or later leaks into the features for *t*.

In [5]:
import math
from collections import deque
from tqdm.auto import tqdm

FEATURES = ["mastery", "log_evidence", "log_depth", "log_prereq_count", "mean_prereq_mastery", "bkt_p_correct"]
BKT_P = FEATURES.index("bkt_p_correct")
F = len(FEATURES)
LEARNER = ["overall_correct_rate", "recent_correct_rate", "previous_correct"]
L = len(LEARNER)
RECENT = 5
# Plain Python lists: much faster than numpy for one value at a time.
P_INIT, P_LEARN, P_GUESS, P_SLIP = (a.tolist() for a in (prior, learn, guess, slip))
LOG_DEPTH = np.log1p(depth).tolist()
LOG_PREREQS = [math.log1p(len(p)) for p in prereqs]

def node_features(v, state, count):
    ps = prereqs[v]
    mean_pm = sum(state[p] for p in ps) / len(ps) if ps else 1.0
    m = state[v]
    return (m, math.log1p(count[v]), LOG_DEPTH[v], LOG_PREREQS[v], mean_pm,
            m * (1 - P_SLIP[v]) + (1 - m) * P_GUESS[v])

n = len(inter)
X_t = np.zeros((n, F), np.float32)
X_n = np.zeros((n, K, F), np.float32)
M = np.zeros((n, K), bool)
X_l = np.zeros((n, L), np.float32)
p_bkt = np.zeros(n, np.float32)

users = inter.user_id.to_numpy()
ex = inter.exercise_idx.to_numpy().tolist()
y = inter.correct.to_numpy().astype(bool)
y_list = y.tolist()
new_student = np.r_[True, users[1:] != users[:-1]].tolist()

for i in tqdm(range(n), mininterval=5):
    if new_student[i]:
        state, count = list(P_INIT), [0] * n_ex
        answered, right, recent, previous = 0, 0, deque(maxlen=RECENT), 0.5
    X_l[i] = ((right + 1) / (answered + 2), (sum(recent) + 1) / (len(recent) + 2), previous)
    v = ex[i]
    X_t[i] = node_features(v, state, count)
    for j, u in enumerate(prereqs[v]):
        X_n[i, j] = node_features(u, state, count)
        M[i, j] = True
    m, g, s = state[v], P_GUESS[v], P_SLIP[v]
    p_bkt[i] = m * (1 - s) + (1 - m) * g
    if y_list[i]:
        post = m * (1 - s) / (m * (1 - s) + (1 - m) * g)
    else:
        post = m * s / (m * s + (1 - m) * (1 - g))
    state[v] = post + (1 - post) * P_LEARN[v]
    count[v] += 1
    answered += 1
    right += y_list[i]
    recent.append(y_list[i])
    previous = float(y_list[i])

fold = inter.user_id.map(fold_of).fillna(-1).astype(int).to_numpy()  # -1 = test student
is_test = np.isin(users, list(test_ids))
assert not (is_test & (fold >= 0)).any()
pd.concat([pd.DataFrame(X_t, columns=FEATURES), pd.DataFrame(X_l, columns=LEARNER)], axis=1).describe().round(3)

  0%|          | 0/5959373 [00:00<?, ?it/s]

,mastery,log_evidence,log_depth,log_prereq_count,mean_prereq_mastery,bkt_p_correct,overall_correct_rate,recent_correct_rate,previous_correct
count,5959373.000,5959373.000,5959373.000,5959373.000,5959373.000,5959373.000,5959373.000,5959373.000,5959373.000
mean,0.845,2.271,2.553,0.778,0.796,0.832,0.828,0.735,0.831
std,0.271,1.309,1.268,0.389,0.256,0.163,0.113,0.164,0.362
min,0.000,0.000,0.000,0.000,0.000,0.000,0.002,0.143,0.000
25%,0.833,1.386,1.792,0.693,0.637,0.801,0.790,0.714,1.000
50%,0.990,2.197,3.045,0.693,0.955,0.880,0.853,0.857,1.000
75%,1.000,3.091,3.497,1.099,1.000,0.947,0.900,0.857,1.000
max,1.000,8.048,4.190,2.079,1.000,1.000,0.996,0.857,1.000


## 3. The model

In [6]:
import torch.nn as nn
import torch.nn.functional as fn

class PrereqGAT(nn.Module):
    """One GAT layer over a node and its direct prerequisites (plus a self-loop), then a classifier
    that corrects BKT: logit = logit(bkt_p_correct of the target) + correction (zero at the start)."""

    def __init__(self, f, l, heads, hidden, classifier, slope):
        super().__init__()
        self.heads, self.hidden, self.slope = heads, hidden, slope
        self.W = nn.Linear(f, heads * hidden, bias=False)
        self.a_dst = nn.Parameter(torch.empty(heads, hidden))
        self.a_src = nn.Parameter(torch.empty(heads, hidden))
        nn.init.xavier_uniform_(self.a_dst)
        nn.init.xavier_uniform_(self.a_src)
        self.fc1 = nn.Linear(heads * hidden + f + l, classifier)  # + learner features
        self.fc2 = nn.Linear(classifier, 1)
        nn.init.zeros_(self.fc2.weight)  # start exactly at BKT
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x_t, x_n, mask, x_l):
        B, k, _ = x_n.shape
        H, D = self.heads, self.hidden
        nodes = torch.cat([x_t.unsqueeze(1), x_n], 1)                    # [B, 1+K, F], slot 0 = self
        h = self.W(nodes).view(B, 1 + k, H, D)
        present = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=mask.device), mask], 1)
        score = (h[:, :1] * self.a_dst).sum(-1) + (h * self.a_src).sum(-1)  # [B, 1+K, H]
        score = fn.leaky_relu(score, self.slope).masked_fill(~present.unsqueeze(-1), float("-inf"))
        alpha = torch.softmax(score, dim=1)
        z = fn.elu((alpha.unsqueeze(-1) * h).sum(1)).reshape(B, H * D)
        base = torch.logit(x_t[:, BKT_P].clamp(1e-4, 1 - 1e-4))
        correction = self.fc2(torch.relu(self.fc1(torch.cat([z, x_t, x_l], -1)))).squeeze(-1)
        return base + correction, alpha

def new_model():
    return PrereqGAT(F, L, CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["CLASSIFIER"], CONFIG["LEAKY_SLOPE"]).to(DEVICE)

print(sum(p.numel() for p in new_model().parameters()), "parameters")

2913 parameters


In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score

T_xt = torch.from_numpy(X_t).to(DEVICE)
T_xn = torch.from_numpy(X_n).to(DEVICE)
T_m = torch.from_numpy(M).to(DEVICE)
T_xl = torch.from_numpy(X_l).to(DEVICE)
T_y = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

def train(rows, seed=SEED):
    torch.manual_seed(seed)
    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    rows = torch.from_numpy(rows).to(DEVICE)
    g = torch.Generator(device="cpu").manual_seed(seed)
    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        order = rows[torch.randperm(len(rows), generator=g).to(DEVICE)]
        total = 0.0
        for b in range(0, len(order), CONFIG["BATCH"]):
            idx = order[b:b + CONFIG["BATCH"]]
            logit, _ = model(T_xt[idx], T_xn[idx], T_m[idx], T_xl[idx])
            loss = fn.binary_cross_entropy_with_logits(logit, T_y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f"  epoch {epoch + 1}: train loss {total / len(order):.4f}")
    return model

@torch.no_grad()
def predict(model, rows):
    model.eval()
    out = []
    for b in range(0, len(rows), 65536):
        idx = torch.from_numpy(rows[b:b + 65536]).to(DEVICE)
        out.append(torch.sigmoid(model(T_xt[idx], T_xn[idx], T_m[idx], T_xl[idx])[0]).cpu().numpy())
    return np.concatenate(out)

def scores(y_true, p):
    return {"auc": roc_auc_score(y_true, p), "accuracy": accuracy_score(y_true, p >= 0.5)}

## 4. Five-fold cross-validation: GAT vs BKT-only on the same students

In [8]:
from scipy.stats import ttest_rel

cv = []
for k in range(5):
    train_rows = np.flatnonzero((fold >= 0) & (fold != k))
    val_rows = np.flatnonzero(fold == k)
    print(f"fold {k}: {len(train_rows):,} training answers, {len(val_rows):,} validation answers")
    model = train(train_rows)
    gat = scores(y[val_rows], predict(model, val_rows))
    bkt_only = scores(y[val_rows], p_bkt[val_rows])
    cv.append({"fold": k, "gat_auc": gat["auc"], "bkt_auc": bkt_only["auc"],
               "gat_accuracy": gat["accuracy"], "bkt_accuracy": bkt_only["accuracy"]})
    print(f"  GAT AUC {gat['auc']:.4f}   BKT AUC {bkt_only['auc']:.4f}")

cv = pd.DataFrame(cv)
t, p = ttest_rel(cv.gat_auc, cv.bkt_auc)
summary_cv = {"gat_auc_mean": cv.gat_auc.mean(), "gat_auc_sd": cv.gat_auc.std(),
              "bkt_auc_mean": cv.bkt_auc.mean(), "bkt_auc_sd": cv.bkt_auc.std(),
              "paired_t": t, "paired_p": p}
display(cv.round(4))
print({k: round(float(v), 4) for k, v in summary_cv.items()})

fold 0: 3,803,701 training answers, 977,405 validation answers
  epoch 1: train loss 0.3657
  epoch 2: train loss 0.3641
  epoch 3: train loss 0.3637
  epoch 4: train loss 0.3633
  epoch 5: train loss 0.3629
  GAT AUC 0.7968   BKT AUC 0.7854
fold 1: 3,803,712 training answers, 977,394 validation answers
  epoch 1: train loss 0.3658
  epoch 2: train loss 0.3641
  epoch 3: train loss 0.3637
  epoch 4: train loss 0.3632
  epoch 5: train loss 0.3629
  GAT AUC 0.7994   BKT AUC 0.7882
fold 2: 3,871,069 training answers, 910,037 validation answers
  epoch 1: train loss 0.3640
  epoch 2: train loss 0.3624
  epoch 3: train loss 0.3619
  epoch 4: train loss 0.3617
  epoch 5: train loss 0.3615
  GAT AUC 0.7925   BKT AUC 0.7817
fold 3: 3,796,280 training answers, 984,826 validation answers
  epoch 1: train loss 0.3654
  epoch 2: train loss 0.3640
  epoch 3: train loss 0.3636
  epoch 4: train loss 0.3633
  epoch 5: train loss 0.3630
  GAT AUC 0.7955   BKT AUC 0.7813
fold 4: 3,849,662 training answe

,fold,gat_auc,bkt_auc,gat_accuracy,bkt_accuracy
0,0,0.7968,0.7854,0.8548,0.8513
1,1,0.7994,0.7882,0.8553,0.8519
2,2,0.7925,0.7817,0.8511,0.8484
3,3,0.7955,0.7813,0.8553,0.8509
4,4,0.7969,0.7858,0.8533,0.8500


{'gat_auc_mean': 0.7962, 'gat_auc_sd': 0.0025, 'bkt_auc_mean': 0.7845, 'bkt_auc_sd': 0.0029, 'paired_t': 18.7453, 'paired_p': 0.0}


## 5. Final model on all training students, scored once on the held-out test students

In [9]:
train_rows = np.flatnonzero(fold >= 0)
test_rows = np.flatnonzero(is_test)
final_model = train(train_rows)
p_gat_test = predict(final_model, test_rows)

majority = float(y[train_rows].mean() >= 0.5)
test_results = pd.DataFrame([
    {"model": "majority class", "auc": 0.5, "accuracy": accuracy_score(y[test_rows], np.full(len(test_rows), majority))},
    {"model": "BKT only", **scores(y[test_rows], p_bkt[test_rows])},
    {"model": f"BKT + GAT ({CONFIG['MODEL_VERSION']})", **scores(y[test_rows], p_gat_test)},
]).round(4)
test_results

  epoch 1: train loss 0.3649
  epoch 2: train loss 0.3632
  epoch 3: train loss 0.3627
  epoch 4: train loss 0.3624
  epoch 5: train loss 0.3621


,model,auc,accuracy
0,majority class,0.5000,0.8342
1,BKT only,0.7804,0.8510
2,BKT + GAT (gat-v2),0.7930,0.8544


## 6. Export for numpy, and prove the numpy version gives the same answers

The service has no PyTorch. `gat_forward_numpy` below is the reference the service copies; it must match PyTorch on test answers.

In [10]:
def export_weights(model):
    sd = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in model.state_dict().items()}
    return {"W": sd["W.weight"].T, "a_dst": sd["a_dst"], "a_src": sd["a_src"],
            "fc1_w": sd["fc1.weight"].T, "fc1_b": sd["fc1.bias"],
            "fc2_w": sd["fc2.weight"].T, "fc2_b": sd["fc2.bias"]}

def gat_forward_numpy(w, x_t, x_n, mask, x_l, heads, hidden, slope):
    """x_t [B,F], x_n [B,K,F], mask [B,K], x_l [B,L] -> (probability [B], attention [B,1+K,H])."""
    B, k, _ = x_n.shape
    nodes = np.concatenate([x_t[:, None], x_n], 1)
    h = (nodes @ w["W"]).reshape(B, 1 + k, heads, hidden)
    score = (h[:, :1] * w["a_dst"]).sum(-1) + (h * w["a_src"]).sum(-1)
    score = np.where(score > 0, score, slope * score)
    present = np.concatenate([np.ones((B, 1), bool), mask], 1)
    score = np.where(present[..., None], score, -np.inf)
    score = score - score.max(1, keepdims=True)
    alpha = np.exp(score) / np.exp(score).sum(1, keepdims=True)
    z = (alpha[..., None] * h).sum(1).reshape(B, heads * hidden)
    z = np.where(z > 0, z, np.expm1(z))
    hidden1 = np.maximum(np.concatenate([z, x_t, x_l], -1) @ w["fc1_w"] + w["fc1_b"], 0)
    p_bkt = np.clip(x_t[:, BKT_P], 1e-4, 1 - 1e-4)
    logit = np.log(p_bkt / (1 - p_bkt)) + (hidden1 @ w["fc2_w"] + w["fc2_b"])[:, 0]
    return 1 / (1 + np.exp(-logit)), alpha

weights = export_weights(final_model)
check = test_rows[:20_000]
p_np, _ = gat_forward_numpy(weights, X_t[check], X_n[check], M[check], X_l[check],
                            CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["LEAKY_SLOPE"])
gap = np.abs(p_np - predict(final_model, check)).max()
print(f"max |numpy - torch| = {gap:.2e}")
assert gap < 1e-4, "numpy export does not reproduce the model"

max |numpy - torch| = 1.79e-07


## 7. Attention and the explanation rule

For answers on exercises that have prerequisites: how much attention goes to prerequisite edges, and which prerequisite the explanation rule would name (argmax of attention x (1 - mastery), attention averaged over heads).

In [11]:
has_prereq = check[M[check].any(1)]
_, alpha = gat_forward_numpy(weights, X_t[has_prereq], X_n[has_prereq], M[has_prereq], X_l[has_prereq],
                             CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["LEAKY_SLOPE"])
edge_attention = alpha[:, 1:].mean(-1)                         # [B, K], mean over heads
weakness = edge_attention * (1 - X_n[has_prereq][..., 0])
weakness[~M[has_prereq]] = -1
named = weakness.argmax(1)
print(f"answers with prerequisites: {len(has_prereq):,}")
print(f"mean attention on prerequisite edges (vs self-loop): {alpha[:, 1:].sum(1).mean():.3f}")
print(f"explanation names the least-mastered prerequisite in "
      f"{(named == np.where(M[has_prereq], X_n[has_prereq][..., 0], 9).argmin(1)).mean():.1%} of cases")

answers with prerequisites: 17,671
mean attention on prerequisite edges (vs self-loop): 0.495
explanation names the least-mastered prerequisite in 96.6% of cases


## 8. Save the model, the feature spec and a manifest

In [12]:
import hashlib, platform, sklearn

model_path = OUT_DIR / f"{CONFIG['MODEL_VERSION']}.npz"
np.savez(model_path, **weights)
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()

feature_spec = {
    "model_version": CONFIG["MODEL_VERSION"],
    "features": FEATURES,
    "learner_features": LEARNER,
    "learner_transforms": {"overall_correct_rate": "(correct + 1) / (answered + 2) over all earlier answers",
                           "recent_correct_rate": f"(correct + 1) / (n + 2) over the last {RECENT} answers",
                           "previous_correct": "1 or 0 for the previous answer, 0.5 before the first"},
    "classifier_input": "concat(attention output, target node features, learner features)",
    "transforms": {"log_evidence": "log1p(count)", "log_depth": "log1p(depth)",
                   "log_prereq_count": "log1p(n_prerequisites)",
                   "mean_prereq_mastery": "mean over direct prerequisites, 1.0 when none",
                   "bkt_p_correct": "mastery * (1 - p_slip) + (1 - mastery) * p_guess, that node's BKT values"},
    "output": "P(correct) = sigmoid(logit(clip(bkt_p_correct of target, 1e-4, 1 - 1e-4)) + correction)",
    "architecture": {"heads": CONFIG["HEADS"], "hidden": CONFIG["HIDDEN"],
                     "classifier": CONFIG["CLASSIFIER"], "leaky_slope": CONFIG["LEAKY_SLOPE"],
                     "self_loop": True, "attention_slot_0": "self", "max_prerequisites_trained": int(K)},
    "explanation": "weak prerequisite = argmax over prerequisite edges of mean-head attention x (1 - mastery)",
    "bkt_params": "junyi_processed/bkt/params.csv (notebook 03)",
    "sha256": model_sha,
    "test_results": test_results.to_dict(orient="records"),
    "cv": {k: float(v) for k, v in summary_cv.items()},
}
(OUT_DIR / "feature_spec.json").write_text(json.dumps(feature_spec, indent=2))
cv.to_csv(OUT_DIR / "cv_folds.csv", index=False)
test_results.to_csv(OUT_DIR / "test_results.csv", index=False)
print("model:", model_path.name, model_sha)

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "scikit-learn": sklearn.__version__, "networkx": nx.__version__}

model: gat-v2.npz 914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d


In [13]:
import hashlib, json, platform
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "created_at": "2026-10-10T14:01:41.074500+00:00",
  "config": {
    "SEED": 42,
    "QUICK": false,
    "HEADS": 4,
    "HIDDEN": 16,
    "CLASSIFIER": 32,
    "EPOCHS": 5,
    "BATCH": 4096,
    "LR": 0.003,
    "WEIGHT_DECAY": 1e-05,
    "LEAKY_SLOPE": 0.2,
    "MODEL_VERSION": "gat-v2"
  },
  "versions": {
    "python": "3.13.15",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "torch": "2.11.0+cu130",
    "scikit-learn": "1.6.1",
    "networkx": "3.7"
  },
  "sha256": {
    "cv_folds.csv": "6186bc90e485c4e65eff99ef658a8c2857acf8b510a25342e384a8e47c5b953a",
    "feature_spec.json": "cff9accd4cf61966121d7feea02bef7281dc6fb713a442b4a3f4fe0cb39ea5e4",
    "gat-v1.npz": "3443983e6287b9e0b033cabfe445ce4faeeaa67b00d60549c2471cf46bb802da",
    "gat-v2.npz": "914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d",
    "test_results.csv": "6b4ffb49b0ee59b1cb49e5af2c50480c0c6ba4e69892ef9be0233a6d0709a704"
  }
}


In [14]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/gat_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

Outputs stored in /content/drive/MyDrive/adaptlearn-c1/junyi_processed/gat


## For the report and the service

- Report the CV table (mean and SD of AUC, paired t-test GAT vs BKT) and the test table against the 83.2% majority class.
- Copy `gat-v2.npz` and `feature_spec.json` to `curriculum-service` on Sat 11 (not into git); the service checks the SHA-256 before loading.
- Report the design change: the first version (5 features, no BKT offset) scored AUC 0.756 against BKT's 0.785; adding `bkt_p_correct` and learning a correction on top of BKT fixed the missing per-exercise guess and slip.
- Report v2: learner-level features (overall and recent correct rate, previous answer) and 5 epochs, added because v1 (test AUC 0.7835) trailed DKT (0.7877) and GKT (0.7885); all features remain graph-agnostic, so the model still transfers to the Java course.
- Notebook 06 reuses this code for the ablations (no attention; similarity graph instead of prerequisites).

Before committing: **Clear all outputs**.